### Imports & Setup

Run the first cell once, then click "Restart Kernel and Run All Cells", (all cells including the first one)

In [ ]:
import os
import sys
import subprocess
from tqdm import tqdm

steps = [
    "Check core library versions",
    "Uninstall numpy, pandas, matplotlib",
    "Install compatible versions",
    "Check/install Plotly",
    "Clone LogView (if needed)",
    "Install LogView requirements",
    "Update sys.path"
]

def run_quiet(cmd):
    try:
        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
    except subprocess.CalledProcessError as e:
        print(f"\nCommand failed: {' '.join(cmd)}")
        print("Please check your environment or run with output enabled for details.")
        raise e

progress = tqdm(steps, desc="Setting up environment", unit="step")

try:
    # Check core versions
    progress.update()
    import numpy as np
    import pandas as pd
    import matplotlib

    versions_ok = (
        np.__version__.startswith("1.24") and
        pd.__version__.startswith("2.1") and
        matplotlib.__version__.startswith("3.7")
    )

    # Reinstall if needed
    if not versions_ok:
        progress.update()
        run_quiet([sys.executable, "-m", "pip", "uninstall", "-y", "numpy", "pandas", "matplotlib"])
        progress.update()
        run_quiet([sys.executable, "-m", "pip", "install", "numpy==1.24.4", "pandas==2.1.4", "matplotlib==3.7.3"])
    else:
        progress.update(2)

    # Check/install plotly
    progress.update()
    try:
        import plotly
    except ImportError:
        run_quiet([sys.executable, "-m", "pip", "install", "plotly"])

    # Clone logview if needed
    progress.update()
    if not os.path.exists("logview"):
        run_quiet(["git", "clone", "https://github.com/fzerbato/logview.git"])

    # Install requirements
    progress.update()
    run_quiet([sys.executable, "-m", "pip", "install", "-r", "logview/requirements.txt"])

    # Add paths
    progress.update()
    cwd = os.getcwd()
    logview_path = os.path.abspath("logview")
    for path in [cwd, logview_path]:
        if path not in sys.path:
            sys.path.append(path)

    progress.close()
    print("Setup complete. Please restart the kernel and run all cells.")
except Exception:
    progress.close()
    print("Setup failed. See the error above.")


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Program Files\Python310\python.exe -m pip install --upgrade pip' command.


In [ ]:
try:
    import zipfile
    import plotly.io as pio
    import pandas as pd
    import pm4py
    from logview.utils import LogViewBuilder
    from logview.predicate import *
    from filter_visualization import query_exploration_icicle, query_breakdown_pie, interactive_icicle, interactive_pie, chart_selecting
    from pm4py.objects.log.importer.xes import importer as xes_importer
    print("All modules successfully imported.")
except Exception as e:
    print(f"Import failed: {e}")
    print("Please restart the kernel and rerun this cell.")

pio.renderers.default = 'notebook'

### Import LogView

In [113]:
if not os.path.exists("logview"):
    subprocess.run(["git", "clone", "https://github.com/fzerbato/logview.git"], check=True)
    print("Cloned logview.")
else:
    print("logview already cloned.")

logview_path = os.path.abspath("logview")

if logview_path not in sys.path:
    sys.path.append(logview_path)
    print(f"Added to sys.path: {logview_path}")

subprocess.run([sys.executable, "-m", "pip", "install", "-r", "logview/requirements.txt"], check=True)


logview already cloned.


CompletedProcess(args=['c:\\Program Files\\Python310\\python.exe', '-m', 'pip', 'install', '-r', 'logview/requirements.txt'], returncode=0)

### Load and Prepare Event Log


In [ ]:
CASE_ID_COL = 'Case ID'
TIMESTAMP_COL = 'Complete Timestamp'
ACTIVITY_COL = 'Activity'

path_to_log = "./dataset/Road_Traffic_Fine_Management_Process.csv"
df = pd.read_csv(path_to_log, dtype={'Resource': str, 'matricola': str}, parse_dates=[TIMESTAMP_COL])
df = df.sort_values([CASE_ID_COL, TIMESTAMP_COL], ignore_index=True)
log = pm4py.format_dataframe(df, case_id=CASE_ID_COL, activity_key=ACTIVITY_COL, timestamp_key=TIMESTAMP_COL)

display(log)

,Case ID,Activity,Resource,Complete Timestamp,Variant,Variant index,amount,article,dismissal,expense,...,notificationType,paymentAmount,points,totalPaymentAmount,vehicleClass,case:concept:name,concept:name,time:timestamp,@@index,@@case_index
0,A1,Create Fine,561,2006-07-24 00:00:00+00:00,Variant 3,3,35.0,157.0,NIL,NaN,...,NaN,NaN,0.0,0.0,A,A1,Create Fine,2006-07-24 00:00:00+00:00,0,0
1,A1,Send Fine,NaN,2006-12-05 00:00:00+00:00,Variant 3,3,NaN,NaN,NaN,11.00,...,NaN,NaN,NaN,NaN,NaN,A1,Send Fine,2006-12-05 00:00:00+00:00,1,0
2,A100,Create Fine,561,2006-08-02 00:00:00+00:00,Variant 1,1,35.0,157.0,NIL,NaN,...,NaN,NaN,0.0,0.0,A,A100,Create Fine,2006-08-02 00:00:00+00:00,2,1
3,A100,Send Fine,NaN,2006-12-12 00:00:00+00:00,Variant 1,1,NaN,NaN,NaN,11.00,...,NaN,NaN,NaN,NaN,NaN,A100,Send Fine,2006-12-12 00:00:00+00:00,3,1
4,A100,Insert Fine Notification,NaN,2007-01-15 00:00:00+00:00,Variant 1,1,NaN,NaN,NaN,NaN,...,P,NaN,NaN,NaN,NaN,A100,Insert Fine Notification,2007-01-15 00:00:00+00:00,4,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
561465,V9999,Create Fine,25,2002-09-07 00:00:00+00:00,Variant 1,1,131.0,142.0,NIL,NaN,...,NaN,NaN,0.0,0.0,A,V9999,Create Fine,2002-09-07 00:00:00+00:00,561465,150369
561466,V9999,Send Fine,NaN,2002-10-25 00:00:00+00:00,Variant 1,1,NaN,NaN,NaN,15.16,...,NaN,NaN,NaN,NaN,NaN,V9999,Send Fine,2002-10-25 00:00:00+00:00,561466,150369
561467,V9999,Insert Fine Notification,NaN,2002-11-04 00:00:00+00:00,Variant 1,1,NaN,NaN,NaN,NaN,...,P,NaN,NaN,NaN,NaN,V9999,Insert Fine Notification,2002-11-04 00:00:00+00:00,561467,150369
561468,V9999,Add penalty,NaN,2003-01-03 00:00:00+00:00,Variant 1,1,262.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,V9999,Add penalty,2003-01-03 00:00:00+00:00,561468,150369


### Instantiating a LogView object ###
The LogViewBuilder class allows instantiating a LogView object, which serves as the central interface for creating LogView instances. It provides a single point of access for interacting with the different framework components.

In [100]:
# Build LogView
log_view = LogViewBuilder.build_log_view(log)

### Apply filters from the LogView tutorial

In [101]:
#  Find all cases that end with activity Payment or Send for Credit Collection
query_t = Query('SCC_and_no_Payment', [EqToConstant('Activity', 'Send for Credit Collection'), NotEqToConstant('Activity', 'Payment')])
result_set_query_t, complement_t = log_view.evaluate_query('rs_SCC_and_no_Payment', log, query_t)

# Cases with activity Send for Credit Collection
query_r = Query('SCC', EqToConstant('Activity', 'Send for Credit Collection'))
result_set_query_r, complement_result_set_query_r = log_view.evaluate_query('rs_SCC', log, query_r)

# Cases with activities Send for Credit Collection and without activity Payment
query_q = Query('SCC_and_no_Payment_2', NotEqToConstant('Activity', 'Payment'))
result_set_query_q, complement_result_set_query_q = log_view.evaluate_query('rs_SCC_and_no_Payment_2', result_set_query_r, query_q)

# Cases without activity Payment
query_s = Query('no_Payment', NotEqToConstant('Activity', 'Payment'))
result_set_query_s, complement_result_set_query_s = log_view.evaluate_query('rs_no_Payment', log, query_s)


In [102]:
summary = log_view.get_summary()

+----+--------------------+----------------------+-------------------------+----------+
|    | source_log         | query                | result_set              | labels   |
|----+--------------------+----------------------+-------------------------+----------|
|  0 | initial_source_log | SCC_and_no_Payment   | rs_SCC_and_no_Payment   | []       |
|  1 | initial_source_log | SCC                  | rs_SCC                  | []       |
|  2 | rs_SCC             | SCC_and_no_Payment_2 | rs_SCC_and_no_Payment_2 | []       |
|  3 | initial_source_log | no_Payment           | rs_no_Payment           | []       |
+----+--------------------+----------------------+-------------------------+----------+
+----+----------------------+------------------------------------------------------------------------------------+
|    | query                | predicates                                                                         |
|----+----------------------+-------------------------------------

### LogView tutorial analysis with icicle chart
Analyzing the filters from the LogView tutorial using the icicle chart provides insights into the relationship between the activities “Send for Credit Collection” and “Payment.” The chart makes it easy to see whether these activities overlap and to what extent. It also highlights how this overlap relates to other dimensions of the process, such as case duration, the number of activities per case, and the time between activities.

In [103]:
query_exploration_icicle('rs_SCC_and_no_Payment_2', log_view, metric='avg_case_duration_seconds', details=False)
query_exploration_icicle('rs_SCC_and_no_Payment_2', log_view, metric='avg_events_per_case', details=False)
query_exploration_icicle('rs_SCC_and_no_Payment_2', log_view, metric='avg_time_between_events', details=False)

### Exploring the variants

While examining the log, I became curious about the meaning of the variants defined in the dataset and whether they related to the earlier analysis. To investigate, I filtered by variant and visualized the results with icicle charts to check for overlaps.

In [ ]:
# Cases of variant 3
query_1 = Query('var_1', EqToConstant('Variant', 'Variant 1'))
result_set_query_v, complement_result_set_query_v = log_view.evaluate_query('rs_SCC_no_Payment_var_1', result_set_query_q, query_1)
query_exploration_icicle('rs_SCC_no_Payment_var_1', log_view, metric='avg_case_duration_seconds', details=False)

# Cases of variant 3
query_2 = Query('var_2', EqToConstant('Variant', 'Variant 2'))
result_set_query_v, complement_result_set_query_v = log_view.evaluate_query('rs_SCC_no_Payment_var_2', result_set_query_q, query_2)
query_exploration_icicle('rs_SCC_no_Payment_var_2', log_view, metric='avg_case_duration_seconds', details=False)

# Cases of variant 3
query_3 = Query('var_3', EqToConstant('Variant', 'Variant 3'))
result_set_query_v, complement_result_set_query_v = log_view.evaluate_query('rs_SCC_no_Payment_var_3', result_set_query_q, query_3)
query_exploration_icicle('rs_SCC_no_Payment_var_3', log_view, metric='avg_case_duration_seconds', details=False)

The analysis suggests that:

- Variant 1 largely corresponds to cases that were sent for credit collection but not paid.

- Variant 2 largely corresponds cases that were not sent for credit collection but were paid. Interestingly, some cases matching this behavior are not labeled as Variant 2, indicating that another variable may play a role in this classification.

- Variant 3 seems to capture cases that were neither sent for credit collection nor paid, though again, not all such cases are flagged as Variant 3.

This implies that while the variants partially align with the previous filters, their exact definitions may depend on additional attributes in the log.

### Performance Analysis

Next, I wanted to check whether the results from the initial analysis were consistent over time, since this log spans 13 years of data. I applied a cutoff at the beginning of 2006 to compare the two periods.

In [114]:
cutoff = datetime(2006, 1, 1, tzinfo=pytz.UTC)

query_before_2006 = Query('before_2006', LessEqualToConstant('Complete Timestamp', cutoff))
result_set_before_2006, _ = log_view.evaluate_query('rs_SCC_before_2006', result_set_query_q, query_before_2006)
query_exploration_icicle('rs_SCC_before_2006', log_view, metric='avg_case_duration_seconds', details=False)

The results show that case durations decreased overall after 2006, with the most notable reduction in cases that were sent for credit collection. Interestingly, whether or not a payment occurred in those credit collection cases does not seem to influence this difference in duration.

Next, I considered focusing my analysis on the long-running cases, since these are likely the most interesting from a time- and cost-saving perspective. To do this, I applied a filter for cases lasting between 50 and 80 million seconds (about 1.6–2.5 years), which I identified earlier as the longest duration range. My goal was to see whether this subset still reflected the broader behaviors in the log, particularly around credit collection and payment.

In [115]:
query_a = Query('a', DurationWithin(50_000_000, 80_000_000))
result_set_a, comp_a = log_view.evaluate_query('rs_a', log, query_a)

query_b = Query('b', EqToConstant('Activity', 'Send for Credit Collection'))
result_set_b, comp_b = log_view.evaluate_query('rs_b', result_set_a, query_b)

query_c = Query('c', NotEqToConstant('Activity', 'Payment'))
result_set_c, comp_c = log_view.evaluate_query('rs_c', result_set_b, query_c)

query_exploration_icicle('rs_c', log_view, metric='avg_events_per_case', details=False)


The analysis shows that, as expected, longer cases generally involve more events, supporting the idea that reducing their duration could lead to significant savings. However, I also observed that there are many shorter-duration cases with a similarly high number of events. Excluding these might not be wise, as they also represent complex cases that could benefit from more in-depth analysis.